In [2]:
# ================================================================
# FTSE 100 — CNN-LSTM + ATTENTION FORECASTING PIPELINE

# Features: Log_Return, SMA_Ratio, RSI_6, Volatility_6,
#           Inflation, Interest_Rate, GDP_Growth
# ================================================================



import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import shap
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv1D, LSTM, Dense,
                                      Dropout, BatchNormalization, Layer)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
import warnings


warnings.filterwarnings('ignore')
tf.random.set_seed(42)
np.random.seed(42)


# Global Style
plt.rcParams.update({
    'figure.dpi'        : 130,
    'font.family'       : 'DejaVu Sans',
    'axes.titlesize'    : 11,
    'axes.labelsize'    : 10,
    'xtick.labelsize'   : 8,
    'ytick.labelsize'   : 8,
    'axes.grid'         : True,
    'grid.alpha'        : 0.3,
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
})
C1, C2, C3, C4 = '#1565C0', '#D32F2F', '#2E7D32', '#F57C00'





# STEP 0 — LOAD
# Monthly FTSE 100 data: 289 rows (Jan 2001 – Jan 2025)
# Columns: Open, High, Low, Close, Volume, Inflation, Interest_Rate, GDP
url = "https://raw.githubusercontent.com/jotheesh2405-bit/DeepL/main/ftse_merged_dataset%20(M).csv"

df = pd.read_csv(url, index_col=0)
df.index = pd.to_datetime(df.index, dayfirst=True)
df.sort_index(inplace=True)


YEAR_START = df.index.year.min()
YEAR_END   = df.index.year.max()
DATE_RANGE = f"{YEAR_START}-{YEAR_END}"


print("FTSE 100 — Monthly Dataset")
print(f"Rows   : {len(df)}")
print(f"Period : {df.index[0].date()} to {df.index[-1].date()}")
print(f"Columns: {df.columns.tolist()}")
print(f"Nulls  :\n{df.isnull().sum().to_string()}")





# STEP 1 — CLEAN
df_clean = df.copy()


# OHLC
for col in ['Open', 'High', 'Low', 'Close']:
    df_clean[col] = df_clean[col].ffill().bfill()


# Volume
df_clean['Volume'] = (
    df_clean['Volume']
    .replace(0, np.nan)
    .fillna(df_clean['Volume'].rolling(6, min_periods=1).mean())
    .ffill().bfill()
)


# OHLC outlier clip
for col in ['Open', 'High', 'Low', 'Close']:
    lo = df_clean[col].quantile(0.005)
    hi = df_clean[col].quantile(0.995)
    df_clean[col] = df_clean[col].clip(lo, hi)

df_clean['High'] = df_clean[['Open', 'High', 'Close']].max(axis=1)
df_clean['Low']  = df_clean[['Open', 'Low',  'Close']].min(axis=1)


# Macro columns — forward-fill monthly values
for col in ['Inflation', 'Interest_Rate', 'GDP']:
    df_clean[col] = df_clean[col].ffill().bfill()


print(f"\nCleaning complete. Nulls remaining: {df_clean.isnull().sum().sum()}")
print(f"Inflation    : {df_clean['Inflation'].min():.1f} – {df_clean['Inflation'].max():.1f}")
print(f"Interest_Rate: {df_clean['Interest_Rate'].min():.2f}% – {df_clean['Interest_Rate'].max():.2f}%")
print(f"GDP (£M)     : {df_clean['GDP'].min():,.0f} – {df_clean['GDP'].max():,.0f}")





# STEP 2 — FEATURE ENGINEERING
# -----------------------------------------------------------------------
# Window sizes chosen for monthly data:
#   SMA_6     = 6-month moving average         (was SMA_20 for daily)
#   RSI_6     = 6-month RSI                    (was RSI_14 for daily)
#   Volatility_6 = 6-month std × √12           (was 20-day std × √252)
#   GDP_Growth = MoM % change of GDP
# -----------------------------------------------------------------------

df_feat = df_clean.copy()

df_feat['Log_Return']    = np.log(df_feat['Close'] / df_feat['Close'].shift(1))

df_feat['SMA_6']         = df_feat['Close'].rolling(6).mean()
df_feat['SMA_Ratio']     = df_feat['Close'] / df_feat['SMA_6']

delta = df_feat['Close'].diff()
gain  = delta.clip(lower=0).rolling(6).mean()
loss  = (-delta.clip(upper=0)).rolling(6).mean()
df_feat['RSI_6']         = (100 - 100 / (1 + gain / (loss + 1e-10))) / 100.0

# Monthly annualization: √12 (NOT √252 which is for trading days)
df_feat['Volatility_6']  = df_feat['Log_Return'].rolling(6).std() * np.sqrt(12)

# Macro features
df_feat['GDP_Growth']    = df_feat['GDP'].pct_change()

df_feat.dropna(inplace=True)


FEAT_COLS = [
    'Log_Return', 'SMA_Ratio', 'RSI_6', 'Volatility_6',
    'Inflation', 'Interest_Rate', 'GDP_Growth'
]

print(f"\nFeature engineering complete. Shape: {df_feat.shape}")
print(df_feat[FEAT_COLS].describe().round(5).to_string())





# STEP 3 — EDA PLOTS


# -- Plot 1 -------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(13, 6),
    gridspec_kw={'height_ratios': [3, 1]}, sharex=True
)
fig.suptitle(f'FTSE 100 — Monthly Close Price & Volume ({DATE_RANGE})',
             fontsize=13, fontweight='bold')
ax1.plot(df_feat.index, df_feat['Close'], color=C1, lw=1.2, marker='o',
         markersize=2.5)
ax1.fill_between(df_feat.index, df_feat['Close'], alpha=0.07, color=C1)
ax1.set_ylabel('Index (Points)')
ax2.bar(df_feat.index, df_feat['Volume'] / 1e9, color=C4, alpha=0.55, width=20)
ax2.set_ylabel('Volume (B)')
ax2.set_xlabel('Date')
plt.tight_layout()
plt.savefig('plot01_timeseries.png', bbox_inches='tight')
plt.close()


# -- Plot 2 -------------------------------------------------------
ret      = df_feat['Log_Return'].dropna()
ret_trim = ret[np.abs(ret) < ret.std() * 4]
xr       = np.linspace(ret_trim.min(), ret_trim.max(), 300)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('FTSE 100 — Monthly Log-Return Distribution Analysis',
             fontsize=13, fontweight='bold')
axes[0].hist(ret_trim, bins=40, color=C1, alpha=0.7, density=True,
             edgecolor='white', lw=0.3)
axes[0].plot(xr, stats.norm.pdf(xr, ret_trim.mean(), ret_trim.std()),
             color=C2, lw=2, label='Normal PDF')
axes[0].set_title('Return + Normal Fit')
axes[0].set_xlabel('Monthly Log Return')
axes[0].legend()
axes[1].hist(df_feat['Close'], bins=40, color=C3, alpha=0.7,
             edgecolor='white', lw=0.3)
axes[1].set_title('Close Price Distribution')
axes[1].set_xlabel('Points')
stats.probplot(ret_trim, dist='norm', plot=axes[2])
axes[2].set_title('QQ Plot — Returns vs Normal')
axes[2].get_lines()[0].set(color=C1, markersize=3, alpha=0.5)
axes[2].get_lines()[1].set(color=C2, lw=2)
plt.tight_layout()
plt.savefig('plot02_distribution.png', bbox_inches='tight')
plt.close()


# -- Plot 3 -------------------------------------------------------
fig, axes = plt.subplots(1, 4, figsize=(14, 5))
fig.suptitle('FTSE 100 — OHLC Outlier Detection (Monthly)',
             fontsize=13, fontweight='bold')
for ax, col, cc in zip(axes, ['Open', 'High', 'Low', 'Close'],
                        [C1, C2, C3, C4]):
    ax.boxplot(df_feat[col].values, patch_artist=True, notch=True,
               boxprops=dict(facecolor=cc, alpha=0.5),
               medianprops=dict(color='black', lw=2),
               whiskerprops=dict(color=cc), capprops=dict(color=cc),
               flierprops=dict(marker='o', markerfacecolor=cc,
                               markersize=3, alpha=0.4))
    ax.set_title(col)
    ax.set_ylabel('Points')
plt.tight_layout()
plt.savefig('plot03_boxplots.png', bbox_inches='tight')
plt.close()


# -- Plot 4 — Correlation heatmap (all 7 features + OHLC) ---------
hmap_cols = FEAT_COLS + ['Open', 'High', 'Low', 'Close']
corr = df_feat[hmap_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
fig, ax = plt.subplots(figsize=(13, 10))
fig.suptitle('FTSE 100 — Feature Correlation Heatmap (Monthly)',
             fontsize=13, fontweight='bold')
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, ax=ax,
            annot_kws={'size': 7}, linewidths=0.4, square=True)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.savefig('plot04_correlation.png', bbox_inches='tight')
plt.close()


# -- Plot 5 — Technical features (last 3 years) -------------------
recent = df_feat[str(YEAR_END - 3):]
fig, axes = plt.subplots(4, 1, figsize=(13, 10), sharex=True)
fig.suptitle(f'Monthly Technical Features — {YEAR_END - 3} to {YEAR_END}',
             fontsize=13, fontweight='bold')
axes[0].plot(recent.index, recent['Close'], color=C1, lw=1.2,
             marker='o', markersize=3)
axes[0].plot(recent.index, recent['SMA_6'], color=C2,
             lw=1.8, ls='--', label='SMA 6')
axes[0].set_ylabel('Close / SMA-6')
axes[0].legend(loc='upper left')
axes[1].bar(recent.index, recent['Log_Return'], color=C3, alpha=0.7, width=20)
axes[1].axhline(0, color='black', lw=0.8)
axes[1].set_ylabel('Log Return')
axes[2].plot(recent.index, recent['RSI_6'] * 100, color=C4, lw=1.4,
             marker='o', markersize=3)
axes[2].axhline(70, color=C2, lw=1, ls='--', label='Overbought')
axes[2].axhline(30, color=C1, lw=1, ls='--', label='Oversold')
axes[2].set_ylabel('RSI 6-Month')
axes[2].legend(loc='upper right')
axes[3].plot(recent.index, recent['Volatility_6'], color='purple', lw=1.3,
             marker='o', markersize=3)
axes[3].set_ylabel('Ann. Volatility\n(Monthly √12)')
axes[3].set_xlabel('Date')
plt.tight_layout()
plt.savefig('plot05_features.png', bbox_inches='tight')
plt.close()


# -- Plot 5b — Macro indicators full history ----------------------
fig, axes = plt.subplots(4, 1, figsize=(13, 11), sharex=True)
fig.suptitle(f'FTSE 100 — Macro Indicators ({DATE_RANGE})',
             fontsize=13, fontweight='bold')
axes[0].plot(df_feat.index, df_feat['Close'], color=C1, lw=1.0)
axes[0].fill_between(df_feat.index, df_feat['Close'],
                     alpha=0.07, color=C1)
axes[0].set_ylabel('FTSE 100 (Points)')
axes[1].plot(df_feat.index, df_feat['Inflation'], color=C2, lw=1.2)
axes[1].fill_between(df_feat.index, df_feat['Inflation'],
                     df_feat['Inflation'].min(), alpha=0.08, color=C2)
axes[1].set_ylabel('CPI Inflation (Index)')
axes[2].plot(df_feat.index, df_feat['Interest_Rate'], color=C3, lw=1.3)
axes[2].fill_between(df_feat.index, df_feat['Interest_Rate'],
                     0, alpha=0.08, color=C3)
axes[2].axhline(df_feat['Interest_Rate'].mean(), color=C3,
                lw=1, ls=':', alpha=0.7,
                label=f"Mean {df_feat['Interest_Rate'].mean():.2f}%")
axes[2].set_ylabel('Interest Rate (%)')
axes[2].legend(fontsize=8)
axes[3].bar(df_feat.index, df_feat['GDP_Growth'] * 100,
            color=C4, alpha=0.7, width=20)
axes[3].axhline(0, color='black', lw=0.8)
axes[3].set_ylabel('GDP Growth (MoM %)')
axes[3].set_xlabel('Date')
plt.tight_layout()
plt.savefig('plot05b_macro.png', bbox_inches='tight')
plt.close()


# -- Plot 6 -------------------------------------------------------
roll_m = df_feat['Close'].rolling(24).mean()    # 24-month rolling (2-year)
roll_s = df_feat['Close'].rolling(24).std()
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
fig.suptitle('FTSE 100 — 24-Month Rolling Statistics',
             fontsize=13, fontweight='bold')
ax1.plot(df_feat.index, df_feat['Close'],
         color=C1, lw=0.9, alpha=0.7, label='Close')
ax1.plot(df_feat.index, roll_m, color=C2, lw=1.8, label='24M Rolling Mean')
ax1.fill_between(df_feat.index, roll_m - roll_s, roll_m + roll_s,
                 alpha=0.15, color=C2, label='±1 Std Dev')
ax1.set_ylabel('Points')
ax1.legend(loc='upper left')
ax2.plot(df_feat.index, roll_s, color=C4, lw=1.2)
ax2.set_ylabel('Std Dev (Points)')
ax2.set_xlabel('Date')
plt.tight_layout()
plt.savefig('plot06_rolling.png', bbox_inches='tight')
plt.close()





# STEP 4 — STATISTICAL ANALYSIS


close_s = df_feat['Close']
diff_s  = close_s.diff().dropna()
log_ret = df_feat['Log_Return'].dropna()

stat_cols = ['Close', 'Log_Return', 'RSI_6', 'Volatility_6',
             'Inflation', 'Interest_Rate', 'GDP_Growth']
desc = df_feat[stat_cols].describe()
desc.loc['skewness'] = df_feat[stat_cols].skew()
desc.loc['kurtosis'] = df_feat[stat_cols].kurt()

adf_raw  = adfuller(close_s,  autolag='AIC')
adf_diff = adfuller(diff_s,   autolag='AIC')
adf_ret  = adfuller(log_ret,  autolag='AIC')

jb_stat, jb_p = stats.jarque_bera(log_ret)
ks_stat, ks_p = stats.kstest(
    log_ret, 'norm',
    args=(log_ret.mean(), log_ret.std())
)

print("  DESCRIPTIVE STATISTICS")
print(desc.round(5).to_string())
print("\n  ADF — Close (Raw)  p =", round(adf_raw[1], 6),  "→ Non-Stationary")
print("  ADF — Close (Diff) p =", round(adf_diff[1], 8), "→ Stationary")
print("  ADF — Log Return   p =", round(adf_ret[1], 8),  "→ Stationary")
print(f"\n  JB stat={jb_stat:.2f}  p={jb_p:.6f} | KS stat={ks_stat:.4f}  p={ks_p:.6f}")
print(f"  Skewness={log_ret.skew():+.4f}  ExKurtosis={log_ret.kurt():+.4f}")


# PLOT 7 — Descriptive stats
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('FTSE 100 — Descriptive Statistics (Monthly Features)',
             fontsize=13, fontweight='bold')

ax = axes[0]
ax.axis('off')
display_cols = ['Close', 'Log_Return', 'Inflation', 'Interest_Rate', 'GDP_Growth']
rows = ['count','mean','std','min','25%','50%','75%','max','skewness','kurtosis']
cell_vals = []
for r in rows:
    row_data = []
    for c in display_cols:
        v = desc.loc[r, c]
        if c == 'Close':
            row_data.append(f'{v:,.2f}')
        elif c in ('Log_Return', 'GDP_Growth'):
            row_data.append(f'{v:.5f}')
        elif c == 'Interest_Rate':
            row_data.append(f'{v:.3f}')
        else:
            row_data.append(f'{v:.4f}')
    cell_vals.append(row_data)

col_labels = ['Close', 'Log Ret', 'Inflation', 'Int. Rate', 'GDP Gr.']
tbl = ax.table(cellText=cell_vals, rowLabels=rows, colLabels=col_labels,
               cellLoc='center', loc='center', bbox=[0, 0, 1, 1])
tbl.auto_set_font_size(False)
tbl.set_fontsize(8)
for j in range(len(col_labels)):
    tbl[0, j].set_facecolor(C1)
    tbl[0, j].set_text_props(color='white', fontweight='bold')
for i in range(1, len(rows) + 1):
    for j in range(len(col_labels)):
        tbl[i, j].set_facecolor('#f0f4ff' if i % 2 == 0 else 'white')
    if rows[i - 1] in ['skewness', 'kurtosis']:
        for j in range(len(col_labels)):
            tbl[i, j].set_facecolor('#fff3e0')
ax.set_title('Summary Statistics — Key Features + Macro', fontsize=10, pad=10)

ax = axes[1]
from sklearn.preprocessing import MinMaxScaler as _MMS
_s = _MMS()
box_data = _s.fit_transform(df_feat[FEAT_COLS].dropna())
bp = ax.boxplot(box_data,
                labels=['LogRet','SMA_R','RSI6','Vol6','Infl','Rate','GDP_Gr'],
                patch_artist=True, notch=True,
                medianprops=dict(color='black', lw=2),
                flierprops=dict(marker='o', markersize=3, alpha=0.4))
bp_colors = [C1, C2, C3, C4, '#6A1B9A', '#00838F', '#EF6C00']
for patch, cc in zip(bp['boxes'], bp_colors):
    patch.set_facecolor(cc); patch.set_alpha(0.55)
for whisker, cc in zip(bp['whiskers'], [c for c in bp_colors for _ in range(2)]):
    whisker.set_color(cc)
ax.set_title('All 7 Features — Normalised Distributions (Min-Max)', fontsize=10)
ax.set_ylabel('Normalised Value [0, 1]')
ax.set_xlabel('Feature')
plt.tight_layout()
plt.savefig('plot07_stat_tests.png', bbox_inches='tight')
plt.close()


# PLOT 8 — ADF + ACF + PACF
fig = plt.figure(figsize=(14, 9))
gs  = fig.add_gridspec(2, 2, hspace=0.4, wspace=0.3)
fig.suptitle('Stationarity Analysis — ADF + ACF + PACF (Monthly)',
             fontsize=13, fontweight='bold')

ax00 = fig.add_subplot(gs[0, 0])
ax00.plot(df_feat.index, close_s, color=C1, lw=0.9, marker='o', markersize=2)
ax00.set_title(f'Raw Close\nADF p={adf_raw[1]:.4f} → Non-Stationary', fontsize=9)
ax00.set_ylabel('Points')

ax01 = fig.add_subplot(gs[0, 1])
ax01.plot(df_feat.index, log_ret, color=C3, lw=0.8, alpha=0.8,
          marker='o', markersize=2)
ax01.axhline(0, color='black', lw=0.8, ls=':')
ax01.set_title(f'Log Returns\nADF p={adf_ret[1]:.6f} → Stationary', fontsize=9)
ax01.set_ylabel('Log Return')

ax10 = fig.add_subplot(gs[1, 0])
plot_acf(log_ret, lags=24, ax=ax10, color=C1, alpha=0.05)
ax10.set_title('ACF — Monthly Log Returns (24 Lags)', fontsize=9)
ax10.set_xlabel('Lag (months)')

ax11 = fig.add_subplot(gs[1, 1])
plot_pacf(log_ret, lags=24, ax=ax11, method='ywm', color=C2, alpha=0.05)
ax11.set_title('PACF — Monthly Log Returns (24 Lags)', fontsize=9)
ax11.set_xlabel('Lag (months)')

plt.savefig('plot08_acf.png', bbox_inches='tight')
plt.close()


# PLOT 9 — Normality
xr = np.linspace(log_ret.min(), log_ret.max(), 300)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle(
    f'Normality Tests — Monthly Log Returns  |  '
    f'JB p={jb_p:.5f}   KS p={ks_p:.5f}   '
    f'Skew={log_ret.skew():+.3f}   ExKurt={log_ret.kurt():+.3f}',
    fontsize=11, fontweight='bold'
)
axes[0].hist(log_ret, bins=40, color=C1, alpha=0.65,
             density=True, edgecolor='white', lw=0.2, label='Observed')
axes[0].plot(xr, stats.norm.pdf(xr, log_ret.mean(), log_ret.std()),
             color=C2, lw=2.5, label='Normal PDF')
axes[0].set_title('Return Distribution vs Normal PDF')
axes[0].set_xlabel('Monthly Log Return')
axes[0].set_ylabel('Density')
axes[0].legend()

x_s  = np.sort(log_ret)
ecdf = np.arange(1, len(x_s) + 1) / len(x_s)
ncdf = stats.norm.cdf(x_s, log_ret.mean(), log_ret.std())
ks_idx = int(np.argmax(np.abs(ecdf - ncdf)))
axes[1].plot(x_s, ecdf, color=C1, lw=1.5, label='Empirical CDF')
axes[1].plot(x_s, ncdf, color=C2, lw=2, ls='--', label='Normal CDF')
axes[1].vlines(x_s[ks_idx], ecdf[ks_idx], ncdf[ks_idx],
               color='red', lw=2.5, label=f'Max KS={ks_stat:.4f}')
axes[1].set_title(f'ECDF vs Normal CDF  (KS p={ks_p:.5f})')
axes[1].set_xlabel('Log Return')
axes[1].legend(fontsize=8)

(osm, osr), (slope, intercept, r) = stats.probplot(log_ret, dist='norm')
axes[2].scatter(osm, osr, color=C1, s=8, alpha=0.5, label='Data quantiles')
axes[2].plot(osm, slope * np.array(osm) + intercept,
             color=C2, lw=2.5, label='Normal reference')
axes[2].set_title(f'QQ Plot  (R²={r**2:.4f})')
axes[2].set_xlabel('Theoretical Quantiles')
axes[2].set_ylabel('Sample Quantiles')
axes[2].legend(fontsize=8)
plt.tight_layout()
plt.savefig('plot09_normality.png', bbox_inches='tight')
plt.close()


# PLOT 10 — Seasonal Decomposition
monthly_c = df_feat['Close'].resample('ME').mean().dropna()
dec = seasonal_decompose(monthly_c, model='multiplicative', period=12)

fig, axes = plt.subplots(4, 1, figsize=(13, 9), sharex=True)
fig.suptitle('FTSE 100 — Multiplicative Seasonal Decomposition (Monthly, Period=12)',
             fontsize=13, fontweight='bold')
components = [dec.observed, dec.trend, dec.seasonal, dec.resid]
labels     = ['Observed', 'Trend', 'Seasonal', 'Residual']
for ax, comp, lbl, cc in zip(axes, components, labels, [C1, C2, C3, C4]):
    clean = comp.dropna()
    ax.plot(clean.index, clean.values, color=cc, lw=1.2)
    ax.fill_between(clean.index, clean.values, clean.mean(),
                    alpha=0.08, color=cc)
    ax.set_ylabel(lbl)
axes[-1].set_xlabel('Date')
plt.tight_layout()
plt.savefig('plot10_seasonal.png', bbox_inches='tight')
plt.close()





# STEP 5 — DATA PREPARATION
# -------------------------------------------------------
# SEQ_LEN=12  →  12-month (1-year) lookback window
# BATCH=16    →  appropriate for ~200 training samples
# -------------------------------------------------------

SEQ_LEN   = 12     # 1-year lookback  (was 60 days)
N_TOTAL   = len(df_feat)
BATCH     = 16     # small batch for small dataset
EPOCHS    = 100    # more epochs with early stopping


TRAIN_END = int(N_TOTAL * 0.75)
VAL_END   = int(N_TOTAL * 0.88)


scaler = StandardScaler()
scaler.fit(df_feat[FEAT_COLS].iloc[:TRAIN_END])
scaled = scaler.transform(df_feat[FEAT_COLS]).astype(np.float32)


X_all, y_all = [], []
for i in range(SEQ_LEN, N_TOTAL):
    X_all.append(scaled[i - SEQ_LEN:i])
    y_all.append(scaled[i, 0])          # target: scaled Log_Return (col 0)

X_all = np.array(X_all, dtype=np.float32)
y_all = np.array(y_all, dtype=np.float32)

tr_end = TRAIN_END - SEQ_LEN
va_end = VAL_END   - SEQ_LEN

X_tr, y_tr = X_all[:tr_end],        y_all[:tr_end]
X_va, y_va = X_all[tr_end:va_end],  y_all[tr_end:va_end]
X_te, y_te = X_all[va_end:],        y_all[va_end:]

n_feat = len(FEAT_COLS)   # 7

print(f"\nMonthly data splits:")
print(f"Feature columns ({n_feat}): {FEAT_COLS}")
print(f"Train : {X_tr.shape}  ({tr_end} months)")
print(f"Val   : {X_va.shape}  ({va_end - tr_end} months)")
print(f"Test  : {X_te.shape}  ({len(X_te)} months)")





# STEP 6 — MODEL

class SelfAttention(Layer):
    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.W = Dense(units)
        self.V = Dense(1)

    def call(self, x):
        score  = self.V(tf.nn.tanh(self.W(x)))
        weight = tf.nn.softmax(score, axis=1)
        return tf.reduce_sum(weight * x, axis=1)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'units': self.units})
        return cfg


inp = Input(shape=(SEQ_LEN, n_feat), name='input')

c = Conv1D(32, kernel_size=3, activation='relu',
           padding='same', name='conv1')(inp)
c = BatchNormalization()(c)
c = Conv1D(16, kernel_size=2, activation='relu',
           padding='same', name='conv2')(c)
c = Dropout(0.3)(c)

l = LSTM(32, return_sequences=True, name='lstm1')(c)
l = Dropout(0.3)(l)
l = LSTM(16, return_sequences=True, name='lstm2')(l)

ctx = SelfAttention(16, name='attention')(l)
ctx = BatchNormalization()(ctx)
ctx = Dropout(0.3)(ctx)

d   = Dense(16, activation='relu')(ctx)
d   = Dense(8,  activation='relu')(d)
out = Dense(1,  activation='linear', name='output')(d)

model = Model(inp, out, name='CNN_LSTM_Attn_FTSE100_Monthly')

model.compile(
    optimizer=Adam(learning_rate=0.001, clipnorm=1.0),
    loss='huber',
    metrics=['mae']
)

model.summary()

callbacks = [
    ReduceLROnPlateau(
        monitor  = 'val_loss',
        factor   = 0.5,
        patience = 10,
        min_lr   = 1e-5,
        verbose  = 1
    ),
    EarlyStopping(
        monitor   = 'val_loss',
        patience  = 20,        # stops if no improvement for 20 epochs
        restore_best_weights = True,
        verbose   = 1
    )
]

history = model.fit(
    X_tr, y_tr,
    validation_data=(X_va, y_va),
    epochs=EPOCHS,
    batch_size=BATCH,
    callbacks=callbacks,
    verbose=1
)

EPOCHS_RAN = len(history.history['loss'])
print(f"\nTraining stopped at epoch {EPOCHS_RAN}")





# STEP 7 — EVALUATE


def inv_scale_log_returns(y_scaled, scaler, n_feat):
    buf = np.zeros((len(y_scaled), n_feat), dtype=np.float32)
    buf[:, 0] = y_scaled
    return scaler.inverse_transform(buf)[:, 0]


y_pred_s    = model.predict(X_te, verbose=0).flatten()
pred_lrs    = inv_scale_log_returns(y_pred_s, scaler, n_feat)
true_lrs    = inv_scale_log_returns(y_te,     scaler, n_feat)

te_start    = VAL_END
base_prices = df_feat['Close'].values[te_start - 1 : te_start - 1 + len(y_te)]
pred_prices = base_prices * np.exp(pred_lrs)
true_prices = base_prices * np.exp(true_lrs)
te_dates    = df_feat.index[te_start : te_start + len(y_te)]

mae  = mean_absolute_error(true_prices, pred_prices)
rmse = np.sqrt(mean_squared_error(true_prices, pred_prices))
mape = np.mean(np.abs((true_prices - pred_prices) / true_prices)) * 100
r2   = r2_score(true_prices, pred_prices)

print("CNN-LSTM+ATTENTION — MONTHLY TEST SET PERFORMANCE")
print(f"  MAE   : {mae:.2f}  points")
print(f"  RMSE  : {rmse:.2f}  points")
print(f"  MAPE  : {mape:.3f} %")
print(f"  R2    : {r2:.4f}")





# STEP 8 — BATCHED MC DROPOUT FORECAST

MC_RUNS       = 50          # more runs for tighter CI at monthly freq
FORECAST_STEPS = 6          # 6 monthly steps = 6-month horizon


def batched_mc_forecast_monthly(model, seed_seq_scaled, df_feat_full,
                                 scaler, n_feat, n_runs, n_steps):
    last_price     = float(df_feat_full['Close'].iloc[-1])
    sequences      = np.tile(seed_seq_scaled[np.newaxis],
                             (n_runs, 1, 1)).astype(np.float32)
    current_prices = np.full(n_runs, last_price, dtype=np.float64)

    # Buffer: need at least 7 values for SMA-6 / RSI-6 / Vol-6
    close_buf = np.tile(
        df_feat_full['Close'].iloc[-25:].values, (n_runs, 1)
    ).astype(np.float64)

    all_prices = np.zeros((n_runs, n_steps), dtype=np.float64)

    # Last known macro values — held constant (slow-changing economic data)
    last_inflation  = float(df_feat_full['Inflation'].iloc[-1])
    last_int_rate   = float(df_feat_full['Interest_Rate'].iloc[-1])
    last_gdp_growth = float(df_feat_full['GDP_Growth'].iloc[-1])

    for step in range(n_steps):
        preds_s  = model(sequences, training=True).numpy().flatten()
        buf      = np.zeros((n_runs, n_feat), dtype=np.float32)
        buf[:, 0] = preds_s
        log_rets  = scaler.inverse_transform(buf)[:, 0]

        current_prices     = current_prices * np.exp(log_rets)
        all_prices[:, step] = current_prices
        close_buf = np.hstack([close_buf[:, 1:],
                               current_prices.reshape(-1, 1)])

        # ---- Recompute monthly technical features --------------------
        # SMA_6
        sma6      = close_buf[:, -6:].mean(axis=1)
        sma_ratio = current_prices / sma6

        # RSI_6 (needs 7 values for 6 diffs)
        log_ch    = np.diff(np.log(close_buf[:, -7:]), axis=1)
        gains_m   = np.where(log_ch > 0, log_ch, 0).mean(axis=1)
        losses_m  = np.where(log_ch < 0, -log_ch, 0).mean(axis=1)
        rs        = gains_m / (losses_m + 1e-10)
        rsi       = (100 - 100 / (1 + rs)) / 100.0

        # Volatility_6 with monthly annualization √12
        log_ret_6 = np.diff(np.log(close_buf[:, -7:]), axis=1)
        vol       = log_ret_6.std(axis=1) * np.sqrt(12)

        # ---- Macro features: constant forward-fill ------------------
        infl_col = np.full(n_runs, last_inflation,  dtype=np.float64)
        rate_col = np.full(n_runs, last_int_rate,   dtype=np.float64)
        gdpg_col = np.full(n_runs, last_gdp_growth, dtype=np.float64)

        # Assemble in same order as FEAT_COLS:
        # [Log_Return, SMA_Ratio, RSI_6, Volatility_6, Inflation, Interest_Rate, GDP_Growth]
        new_raw    = np.column_stack(
            [log_rets, sma_ratio, rsi, vol,
             infl_col, rate_col, gdpg_col]
        ).astype(np.float32)
        new_scaled = scaler.transform(new_raw).astype(np.float32)
        sequences  = np.concatenate(
            [sequences[:, 1:, :], new_scaled[:, np.newaxis, :]], axis=1
        )

    return all_prices


print(f"\nRunning MC forecast: {MC_RUNS} runs × {FORECAST_STEPS} monthly steps...")
print(f"Macro held constant → Inflation={df_feat['Inflation'].iloc[-1]:.1f}  "
      f"Rate={df_feat['Interest_Rate'].iloc[-1]:.2f}%  "
      f"GDP_Growth={df_feat['GDP_Growth'].iloc[-1]*100:.4f}%")

seed_seq     = scaled[-SEQ_LEN:].copy()
price_matrix = batched_mc_forecast_monthly(
    model, seed_seq, df_feat, scaler, n_feat, MC_RUNS, FORECAST_STEPS
)

fc_median = np.median(price_matrix, axis=0)
fc_p10    = np.percentile(price_matrix, 10, axis=0)
fc_p25    = np.percentile(price_matrix, 25, axis=0)
fc_p75    = np.percentile(price_matrix, 75, axis=0)
fc_p90    = np.percentile(price_matrix, 90, axis=0)

# Monthly forecast dates
fc_dates = pd.date_range(
    start=df_feat.index[-1] + pd.DateOffset(months=1),
    periods=FORECAST_STEPS,
    freq='MS'         # Month Start — matches dataset frequency
)
df_fc = pd.DataFrame({
    'Median': fc_median, 'P10': fc_p10, 'P25': fc_p25,
    'P75': fc_p75, 'P90': fc_p90
}, index=fc_dates)

print("\n6-Month Forecast — Monthly Median (Points):")
print(df_fc['Median'].round(1).to_string())





# STEP 9 — SHAP + RANDOM FOREST


df_rf = df_feat[FEAT_COLS].copy()
df_rf['Target'] = df_rf['Log_Return'].shift(-1)
df_rf.dropna(inplace=True)

X_rf = df_rf[FEAT_COLS].values
y_rf = df_rf['Target'].values
rf_tr = int(len(X_rf) * 0.80)

rf = RandomForestRegressor(
    n_estimators=300, max_depth=6,
    min_samples_leaf=5, random_state=42, n_jobs=-1
)
rf.fit(X_rf[:rf_tr], y_rf[:rf_tr])
rf_r2 = r2_score(y_rf[rf_tr:], rf.predict(X_rf[rf_tr:]))

print(f"\nRandom Forest R2 (1-month return): {rf_r2:.4f}")

explainer = shap.TreeExplainer(rf)
shap_samp = X_rf[rf_tr:]                  # use all test samples (small dataset)
shap_vals = explainer.shap_values(shap_samp)
mean_shap = np.abs(shap_vals).mean(axis=0)
rf_imp    = rf.feature_importances_



# STEP 10 — OUTPUT PLOTS


# -- Plot 11 -------------------------------------------------------
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(df_feat.index[SEQ_LEN : SEQ_LEN + tr_end],
        df_feat['Close'].values[SEQ_LEN : SEQ_LEN + tr_end],
        color=C1, lw=1.2, marker='o', markersize=2, label='Train (75%)')
ax.plot(df_feat.index[SEQ_LEN + tr_end : SEQ_LEN + va_end],
        df_feat['Close'].values[SEQ_LEN + tr_end : SEQ_LEN + va_end],
        color=C3, lw=1.2, marker='o', markersize=2, label='Validation (13%)')
ax.plot(df_feat.index[SEQ_LEN + va_end:],
        df_feat['Close'].values[SEQ_LEN + va_end:],
        color=C4, lw=1.2, marker='o', markersize=2, label='Test (12%)')
ax.set_title('FTSE 100 — Monthly Train / Validation / Test Split')
ax.set_ylabel('Close (Points)')
ax.set_xlabel('Date')
ax.legend(loc='upper left')
plt.tight_layout()
plt.savefig('plot11_split.png', bbox_inches='tight')
plt.close()


# -- Plot 12 -------------------------------------------------------
ep = range(1, EPOCHS_RAN + 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle(f'CNN-LSTM+Attention — Training Diagnostics ({EPOCHS_RAN} Epochs, EarlyStopping)',
             fontsize=12, fontweight='bold')
axes[0].plot(ep, history.history['loss'],
             color=C1, lw=2, label='Train Loss')
axes[0].plot(ep, history.history['val_loss'],
             color=C2, lw=2, ls='--', label='Val Loss')
axes[0].set_title('Huber Loss (Log Scale)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_yscale('log')
axes[0].legend()
lr_history = history.history.get('learning_rate', [])
if lr_history:
    axes[1].plot(ep, lr_history, color=C4, lw=2)
    axes[1].set_title('Learning Rate — ReduceLROnPlateau')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Learning Rate')
    axes[1].set_yscale('log')
else:
    axes[1].text(0.5, 0.5, 'LR history not available',
                 ha='center', va='center', transform=axes[1].transAxes)
plt.tight_layout()
plt.savefig('plot12_loss.png', bbox_inches='tight')
plt.close()


# -- Plot 13 -------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    f'Monthly Test Set  |  MAE={mae:.0f}  RMSE={rmse:.0f}  '
    f'MAPE={mape:.3f}%  R2={r2:.4f}',
    fontsize=11, fontweight='bold'
)
axes[0].plot(te_dates, true_prices, color=C1, lw=1.5,
             marker='o', markersize=5, label='Actual')
axes[0].plot(te_dates, pred_prices, color=C2, lw=1.5,
             marker='s', markersize=4, ls='--', alpha=0.85, label='Predicted')
axes[0].set_ylabel('Close (Points)')
axes[0].set_xlabel('Date')
axes[0].set_title('Actual vs Predicted Monthly Close Price')
axes[0].legend()
residuals = true_prices - pred_prices
axes[1].bar(te_dates, residuals, color=np.where(residuals >= 0, C3, C2),
            alpha=0.75, width=20)
axes[1].axhline(0, color='black', lw=1.5, ls='--')
axes[1].axhline(residuals.mean(), color=C4, lw=1.5, ls='-.',
                label=f'Mean = {residuals.mean():.1f}')
axes[1].set_title('Monthly Residuals Over Time')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Residual (Points)')
axes[1].legend()
plt.tight_layout()
plt.savefig('plot13_predictions.png', bbox_inches='tight')
plt.close()


# -- Plot 14 — 6-Month Monthly Forecast ---------------------------
hist_w = df_feat['Close'].iloc[-36:]    # last 3 years = 36 months
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(hist_w.index, hist_w.values, color=C1, lw=1.8,
        marker='o', markersize=4, label='Historical (Last 3 Years)')
ax.fill_between(df_fc.index, df_fc['P10'], df_fc['P90'],
                color=C2, alpha=0.12, label='80% Confidence Band')
ax.fill_between(df_fc.index, df_fc['P25'], df_fc['P75'],
                color=C2, alpha=0.25, label='50% Confidence Band')
ax.plot(df_fc.index, df_fc['Median'], color=C2, lw=2.2,
        ls='--', marker='D', markersize=6, label='Median Forecast')
for i, (date, val) in enumerate(zip(df_fc.index, df_fc['Median'])):
    ax.annotate(f'{val:.0f}', (date, val),
                textcoords='offset points', xytext=(0, 10),
                ha='center', fontsize=7.5, color=C2, fontweight='bold')
ax.axvline(df_feat.index[-1], color='gray', lw=1.5, ls=':',
           label='Forecast Start')
ax.set_title('FTSE 100 — 6-Month MC Dropout Forecast (Monthly)',
             fontsize=12, fontweight='bold')
ax.set_ylabel('Close (Points)')
ax.set_xlabel('Date')
ax.legend(loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig('plot14_forecast.png', bbox_inches='tight')
plt.close()


# -- Plot 15 -------------------------------------------------------
fig = plt.figure(figsize=(10, 6))
shap.summary_plot(shap_vals, shap_samp, feature_names=FEAT_COLS,
                  show=False, max_display=len(FEAT_COLS))
plt.title('SHAP — Feature Impact on 1-Month Log Return Prediction',
          fontsize=11, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('plot15_shap.png', bbox_inches='tight')
plt.close()


# -- Plot 16 -------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Feature Interpretability — SHAP vs Random Forest (Monthly, 7 Features)',
             fontsize=13, fontweight='bold')
si = np.argsort(mean_shap)[::-1]
b1 = axes[0].bar(range(n_feat), mean_shap[si], color=C1,
                 alpha=0.85, edgecolor='white')
axes[0].set_xticks(range(n_feat))
axes[0].set_xticklabels([FEAT_COLS[i] for i in si], rotation=30, ha='right')
axes[0].set_title('Mean Absolute SHAP Value')
axes[0].set_ylabel('Mean |SHAP|')
for bar, val in zip(b1, mean_shap[si]):
    axes[0].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() * 1.02,
                 f'{val:.5f}', ha='center', va='bottom', fontsize=7)
ri = np.argsort(rf_imp)[::-1]
b2 = axes[1].bar(range(n_feat), rf_imp[ri], color=C3,
                 alpha=0.85, edgecolor='white')
axes[1].set_xticks(range(n_feat))
axes[1].set_xticklabels([FEAT_COLS[i] for i in ri], rotation=30, ha='right')
axes[1].set_title('Random Forest Importance')
axes[1].set_ylabel('Importance Score')
for bar, val in zip(b2, rf_imp[ri]):
    axes[1].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() * 1.02,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=7)
plt.tight_layout()
plt.savefig('plot16_importance.png', bbox_inches='tight')
plt.close()

FTSE 100 — Monthly Dataset
Rows   : 289
Period : 2001-01-01 to 2025-01-01
Columns: ['Open', 'High', 'Low', 'Close', 'Volume', 'Inflation', 'Interest_Rate', 'GDP']
Nulls  :
Open             0
High             0
Low              0
Close            0
Volume           0
Inflation        0
Interest_Rate    0
GDP              0

Cleaning complete. Nulls remaining: 0
Inflation    : 73.5 – 135.1
Interest_Rate: 0.05% – 5.92%
GDP (£M)     : 493,575 – 703,373

Feature engineering complete. Shape: (283, 14)
       Log_Return  SMA_Ratio      RSI_6  Volatility_6  Inflation  Interest_Rate  GDP_Growth
count   283.00000  283.00000  283.00000     283.00000  283.00000      283.00000   283.00000
mean      0.00139    1.00360    0.57650       0.12102   97.32933        2.14693     0.00138
std       0.03853    0.04699    0.23611       0.05518   16.09925        2.07921     0.01668
min      -0.14858    0.81659    0.00000       0.01427   74.50000        0.05000    -0.19906
25%      -0.01903    0.98464    0.40481

Model: "CNN_LSTM_Attn_FTSE100_Monthly"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (None, 12, 7)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1 (Conv1D)                  │ (None, 12, 32)         │           704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 12, 32)         │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2 (Conv1D)                  │ (None, 12, 16)         │         1,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 12, 16)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm1 (LSTM)                    │ (None, 12, 32)         │         6,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 12, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm2 (LSTM)                    │ (None, 12, 16)         │         3,136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ attention (SelfAttention)       │ (None, 16)             │           289 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 16)             │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,050 (47.07 KB)

 Trainable params: 11,954 (46.70 KB)

 Non-trainable params: 96 (384.00 B)

Epoch 1/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 9s 74ms/step - loss: 0.4678 - mae: 0.8480 - val_loss: 0.4318 - val_mae: 0.7901 - learning_rate: 0.0010
Epoch 2/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 0.4205 - mae: 0.7767 - val_loss: 0.4306 - val_mae: 0.7868 - learning_rate: 0.0010
Epoch 3/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.4405 - mae: 0.8132 - val_loss: 0.4300 - val_mae: 0.7853 - learning_rate: 0.0010
Epoch 4/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.4291 - mae: 0.7942 - val_loss: 0.4295 - val_mae: 0.7853 - learning_rate: 0.0010
Epoch 5/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.4202 - mae: 0.7880 - val_loss: 0.4286 - val_mae: 0.7848 - learning_rate: 0.0010
Epoch 6/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 0.4101 - mae: 0.7795 - val_loss: 0.4271 - val_mae: 0.7829 - learning_rate: 0.0010
Epoch 7/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 0.3988 - mae: 0.7492 - val_loss: 0.4266 - val_mae: 0.7810 - learning_rate: 0.0010
Epoch 